In [1]:
import pandas as pd
import numpy as np
from scipy import stats
# import gc
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from datetime import date
from collections import Counter
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rohit\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [2]:
import os
os.getcwd()

'd:\\workspace\\git_projects\\Power-of-Patients-Capstone\\phenotypes\\scripts'

### Load Data

In [3]:
notes= pd.read_csv('../data/raw/additional_notes.csv')
df_tbi_incident = pd.read_csv('../data/raw/tbi_incident.csv')
new_resulting_factors = pd.read_csv('../data/raw/new_resulting_factors.csv')
patient_info = pd.read_csv('../data/raw/patient_info.csv')

C:\Users\rohit\AppData\Local\Temp\ipykernel_7656\3697455665.py:3: DtypeWarning: Columns (0,4,12) have mixed types. Specify dtype option on import or set low_memory=False.
  new_resulting_factors = pd.read_csv('../data/raw/new_resulting_factors.csv')


# Patient Info

In [4]:
patient_info.head(2)

,patient_id,first_name,last_name,date_of_birth,gender,patient_type,external_id,patient_sub_type
0,5c96ba1a-8b2d-49bc-8e8e-b07761948286,Robin,Lopez,1973-05-23,female,caregiver,NaN,NaN
1,eda39327-b38f-41de-a46a-8782787369b7,Justin,Macks,1991-07-10,male,caregiver,NaN,NaN


In [5]:
patient_info.shape, patient_info.columns

((1240, 8),
 Index(['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender',
        'patient_type', 'external_id', 'patient_sub_type'],
       dtype='object'))

In [6]:
patient_info = patient_info[['patient_id', 'date_of_birth', 'gender',
       'patient_type', 'external_id']]


In [7]:
patient_info.isna().sum()

patient_id          0
date_of_birth       0
gender              0
patient_type        4
external_id      1222
dtype: int64

# Cleaning Additional Notes

In [8]:
notes.head(2)

,id,patient_id,note,logged_at,additional_notes_date
0,2,2bb4475e-7f93-4424-bcff-f29e770f039e,"Stress, travel, too many things going on",2020-10-24 14:40:58,2020-10-24
1,3,71599994-64ed-4db4-8b38-1b7cd4ec2024,Nothing,2020-10-24 16:36:29,2020-10-24


In [9]:
notes.shape, notes.columns

((2134, 5),
 Index(['id', 'patient_id', 'note', 'logged_at', 'additional_notes_date'], dtype='object'))

In [10]:
notes['words'] = notes['note'].str.split()
#remove if 1 word or none, will not contain enough info
#usually 'NO' 'N/A', 'blah', etc..
notes = notes[notes.words.str.len() > 1]
notes = notes.groupby('patient_id').agg(lambda x: x.tolist())

In [11]:
notes1 = []
notes2 = []
notes3 = []
date1 = []
date2 = []
date3 = []

for index, row in notes.iterrows():
    patient_notes = list(sorted(row['note'], key = len, reverse=True))
    if len(row['note']) > 0:
        notes1.append(patient_notes[0])
        date1.append(row['additional_notes_date'][0])
    else:
        notes1.append(np.nan)
        date1.append(np.nan)
    if len(row['note']) > 1:
        notes2.append(patient_notes[1])
        date2.append(row['additional_notes_date'][1])
    else:
        notes2.append(np.nan)
        date2.append(np.nan)
    if len(row['note']) > 2:
        notes3.append(patient_notes[2])
        date3.append(row['additional_notes_date'][2])
    else:
        notes3.append(np.nan)
        date3.append(np.nan)

notes['note1'] = notes1
notes['note2'] = notes2
notes['note3'] = notes3
notes['note1_post_date'] = date1
notes['note2_post_date'] = date2
notes['note3_post_date'] = date3
notes['patient_id'] = notes.index

In [12]:
notes.head(2)

,id,note,logged_at,additional_notes_date,words,note1,note2,note3,note1_post_date,note2_post_date,note3_post_date,patient_id
patient_id,,,,,,,,,,,,
01cf90d6-4f6d-4a34-942b-3e43f9645c1f,"[1572, 1582, 1589]",[Currently dealing with high amount of stress ...,"[2022-02-10 20:39:33, 2022-02-12 03:10:30, 202...","[2022-02-10, 2022-02-12, 2022-02-14]","[[Currently, dealing, with, high, amount, of, ...",Currently dealing with high amount of stress a...,"Felt terrible on initial awakening, then bette...",pajama day. Lots of logistics,2022-02-10,2022-02-12,2022-02-14,01cf90d6-4f6d-4a34-942b-3e43f9645c1f
0335b765-7448-459f-b665-6e79e8e41218,"[1603, 1606, 1612, 1615]","[Anger , have barely gotten out of bed in 3 dy...","[2022-02-22 16:10:37, 2022-02-23 17:22:55, 202...","[2022-02-22, 2022-02-23, 2022-02-26, 2022-02-26]","[[Anger, ,, have, barely, gotten, out, of, bed...",Unable to function at work. I cant even unders...,"After going to the store, I am back in bed due...","Anger , have barely gotten out of bed in 3 dyas",2022-02-22,2022-02-23,2022-02-26,0335b765-7448-459f-b665-6e79e8e41218


In [13]:
sid = SentimentIntensityAnalyzer()

In [14]:
note1_score = []
note2_score = []
note3_score = []
for index, row in notes.iterrows():
    if type(row['note1']) == str:
        note1_score.append(sid.polarity_scores(row['note1'])['compound'])
    else:
        note1_score.append(np.nan)
    if type(row['note2']) == str:
        note2_score.append(sid.polarity_scores(row['note2'])['compound'])
    else:
        note2_score.append(np.nan)
    if type(row['note3']) == str:
        note3_score.append(sid.polarity_scores(row['note3'])['compound'])
    else:
        note3_score.append(np.nan)
notes['note1_score'] = note1_score
notes['note2_score'] = note2_score
notes['note3_score'] = note3_score

In [15]:
final_notes = notes[['patient_id','note1', 'note1_score', 'note2', 'note2_score', 'note3', 'note3_score', 'note1_post_date', 'note2_post_date', 'note3_post_date']]
final_notes.head()

,patient_id,note1,note1_score,note2,note2_score,note3,note3_score,note1_post_date,note2_post_date,note3_post_date
patient_id,,,,,,,,,,
01cf90d6-4f6d-4a34-942b-3e43f9645c1f,01cf90d6-4f6d-4a34-942b-3e43f9645c1f,Currently dealing with high amount of stress a...,-0.7184,"Felt terrible on initial awakening, then bette...",-0.0516,pajama day. Lots of logistics,0.0000,2022-02-10,2022-02-12,2022-02-14
0335b765-7448-459f-b665-6e79e8e41218,0335b765-7448-459f-b665-6e79e8e41218,Unable to function at work. I cant even unders...,0.0000,"After going to the store, I am back in bed due...",-0.3612,"Anger , have barely gotten out of bed in 3 dyas",-0.5719,2022-02-22,2022-02-23,2022-02-26
05d558eb-39fb-484c-be18-a0f0f9ba9de8,05d558eb-39fb-484c-be18-a0f0f9ba9de8,I have problems with choices,-0.4019,NaN,NaN,NaN,NaN,2021-04-28,NaN,NaN
07090be5-41d1-44ba-b62d-2fafeee31cd8,07090be5-41d1-44ba-b62d-2fafeee31cd8,Severely depressed,-0.7430,NaN,NaN,NaN,NaN,2021-03-08,NaN,NaN
08c78b75-b671-4b56-a988-40512461ff36,08c78b75-b671-4b56-a988-40512461ff36,Think I put Friday’s into saturday,0.0000,NaN,NaN,NaN,NaN,2021-03-05,NaN,NaN


# Cleaning TBI Incidents

In [16]:
df_tbi_incident = df_tbi_incident.iloc[:, :9]

In [17]:
df_tbi_incident["injury_from"].value_counts().head(6)

injury_from
Accident     385
Fall         159
Collision    104
Sports        82
Assault       77
Stroke        11
Name: count, dtype: int64

In [18]:
df_tbi_incident_ = df_tbi_incident[df_tbi_incident["describe_event"].notna()]
df_tbi_incident_["injury_from"] = df_tbi_incident_["injury_from"].str.strip()

Counter(" ".join(df_tbi_incident_["describe_event"]).split()).most_common(100);
Counter(" ".join(df_tbi_incident_["describe_event"][df_tbi_incident_["injury_from"] == "Accident"]).split()).most_common(100);
Counter(" ".join(df_tbi_incident_["describe_event"][df_tbi_incident_["injury_from"] == "Assault"]).split()).most_common(100);

collision = [
    "car", "truck", "suv", "van", "vehicle", "motorcycle", "driver", "speeding", "crash",
    "collision", "rear-ended", "head-on", "t-bone", "wreck", "run over", "road accident", 
    "hit by car", "traffic", "auto", "bike accident", "hit-and-run"
]

fall = [
    "fell", "fall", "dropped", "tripped", "tumble", "stumble", "trip", "topple", "collapse",
    "slipped", "slip", "roof", "ladder", "stairs", "tree", "balcony", "height", "scaffold",
    "wet floor", "icy", "uneven surface", "rug", "floor mat"
]

assault = [
    "assault", "assaulted", "attacked", "attack", "punched", "hit", "kicked", "struck", 
    "smacked", "slapped", "beaten", "beating", "fight", "fighting", "rape", "raped",
    "sexual", "sex", "molested", "harassed", "abused", "domestic", "violence", "mugged"
]

accident = ["fell_on","accident", "fire", "cut", "laceration", "choking", "burn", "electrocution"]  # work on progress

sports = [
    "soccer", "football", "basketball", "baseball", "volleyball", "hockey", "rugby", "wrestling",
    "boxing", "martial arts", "karate", "taekwondo", "judo", "running", "track", "field", 
    "pole vault", "swim", "swimming", "diving", "surfing", "ski", "skiing", "snowboarding",
    "snowboard", "skateboarding", "skating", "kayaking", "canoeing", "rowing", "dodgeball", 
    "tennis", "badminton", "pickleball", "gymnastics", "cheerleading", "crossfit", "weightlifting"
]


df_tbi_incident["injury_from_new"] = 0 
df_tbi_incident = df_tbi_incident[df_tbi_incident["describe_event"].notna()]
df_tbi_incident = df_tbi_incident[df_tbi_incident["injury_from"].notna()]
for item in df_tbi_incident["describe_event"].index:
    if any(word in df_tbi_incident["describe_event"][item] for word in collision):
        df_tbi_incident["injury_from_new"][item] = "Colision"
    elif any(word in df_tbi_incident["describe_event"][item] for word in fall):
        df_tbi_incident["injury_from_new"][item] = "Fall"
    elif any(word in df_tbi_incident["describe_event"][item] for word in assault):
        df_tbi_incident["injury_from_new"][item] = "Assault"
    elif any(word in df_tbi_incident["describe_event"][item] for word in sports):
        df_tbi_incident["injury_from_new"][item] = "Sports"
    else:
        df_tbi_incident["injury_from_new"][item] = "Accident"

C:\Users\rohit\AppData\Local\Temp\ipykernel_7656\843835663.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tbi_incident_["injury_from"] = df_tbi_incident_["injury_from"].str.strip()
C:\Users\rohit\AppData\Local\Temp\ipykernel_7656\843835663.py:50: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_i

In [19]:
df_tbi_incident.head(2)

,id,tbi_incident_date,describe_event,injury_from,head_hit_location,patient_id,total_tbi,immediate_symptoms_resulting,num_head_hit_location,injury_from_new
0,1080,7/31/2013,I was riding a longboard to my friends house w...,Fall,Left Side of Head,efd7682b-adde-4b6f-b2d7-16f50597c984,1,"Loss of Consciousness,Disorientation,Incoheren...",NaN,Accident
1,1164,4/3/2024,"On april 3, i was preforming my work duties . ...",Accident,Left Side of Head,bb8e0677-ddc8-49dd-be08-ab6f10ad43f3,1,"Disorientation,Confusion,Dazed or Vacant Stare",NaN,Assault


In [20]:
df_tbi_incident['total_tbi'] = pd.to_numeric(df_tbi_incident['total_tbi'], errors='coerce')

# Drop rows with NaN (i.e., non-numeric entries)
df_tbi_incident = df_tbi_incident.dropna(subset=['total_tbi'])

In [21]:
df_tbi_incident = df_tbi_incident[df_tbi_incident['total_tbi']<= 800]
df_tbi_incident.shape

(936, 10)

In [22]:
##Reading the dataset
df_incident_head_hit_location = pd.read_csv('../data/raw/incident_head_hit_location.csv')
df_incident_head_hit_location.shape

(1806, 3)

In [23]:
df_incident_head_hit_location.head()

,id,head_hit_location,patient_id
0,0,Front of Head,52e19a89-140a-41a4-a3dc-002ebef77f96
1,1,Left Side of Head,c7143c39-e497-4afd-9a17-4572e13a1fca
2,2,Top of Head,c7143c39-e497-4afd-9a17-4572e13a1fca
3,3,Back of Head,417a14f7-b224-4b8d-89aa-4fbb51f630ca
4,4,Top of Head,d6a8454b-c7ae-43c7-b972-436d5dea1b3c


In [24]:
df_incident_head_hit_location['head_hit_location'] = df_incident_head_hit_location['head_hit_location'].astype(str)
head_hit_group = df_incident_head_hit_location[['head_hit_location', 'patient_id']].groupby(['patient_id'], as_index=False).agg({'head_hit_location': ','.join})

In [25]:
head_hit_group = head_hit_group.rename({'head_hit_location': 'all_head_hit_locations'}, axis=1) 
head_hit_group.head()

,patient_id,all_head_hit_locations
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,Top of Head
1,00469456-99a5-4c99-aa48-a918986c7c45,Right Side of Head
2,012bfc75-70c1-4007-ab9e-1f4ee45bd537,All
3,012eccde-f0b9-4ae2-aebf-3c821c461545,"Left Side of Head,Left Side of Head,Left Side ..."
4,01786d46-2829-42e2-8d50-135b7e212ea3,Back of Head


In [26]:
df_tbi_incident_ext = df_tbi_incident.merge(head_hit_group, how='left', on='patient_id')
df_tbi_incident_ext.head(2)

,id,tbi_incident_date,describe_event,injury_from,head_hit_location,patient_id,total_tbi,immediate_symptoms_resulting,num_head_hit_location,injury_from_new,all_head_hit_locations
0,1080,7/31/2013,I was riding a longboard to my friends house w...,Fall,Left Side of Head,efd7682b-adde-4b6f-b2d7-16f50597c984,1.0,"Loss of Consciousness,Disorientation,Incoheren...",NaN,Accident,"Left Side of Head,Left Side of Head"
1,1164,4/3/2024,"On april 3, i was preforming my work duties . ...",Accident,Left Side of Head,bb8e0677-ddc8-49dd-be08-ab6f10ad43f3,1.0,"Disorientation,Confusion,Dazed or Vacant Stare",NaN,Assault,Left Side of Head


In [27]:
df_tbi_incident_ext['head_hit_location_new'] = np.where(df_tbi_incident_ext['head_hit_location'].isna(), df_tbi_incident_ext['all_head_hit_locations'], df_tbi_incident_ext['head_hit_location'])

In [28]:
##Reading the immediate symptoms dataset
df_immediate_symptoms_resulting = pd.read_csv('../data/raw/immediate_symptoms_resulting.csv')
df_immediate_symptoms_resulting.shape

(3460, 3)

In [29]:
df_immediate_symptoms_resulting.head()

,id,immediate_symptoms_resulting,patient_id
0,0,Disorientation,52e19a89-140a-41a4-a3dc-002ebef77f96
1,1,Incoherent Speech,c7143c39-e497-4afd-9a17-4572e13a1fca
2,2,Disorientation,c7143c39-e497-4afd-9a17-4572e13a1fca
3,3,Loss of Consciousness,417a14f7-b224-4b8d-89aa-4fbb51f630ca
4,4,Disorientation,417a14f7-b224-4b8d-89aa-4fbb51f630ca


In [30]:
df_immediate_symptoms_resulting.shape

(3460, 3)

In [31]:
df_immediate_symptoms_resulting['immediate_symptoms_resulting'] = df_immediate_symptoms_resulting['immediate_symptoms_resulting'].astype(str)
symptoms_group = df_immediate_symptoms_resulting[['immediate_symptoms_resulting', 'patient_id']].groupby(['patient_id'], as_index=False).agg({'immediate_symptoms_resulting': ','.join})

In [32]:
symptoms_group = symptoms_group.rename({'immediate_symptoms_resulting': 'immediate_symptoms_resulting_ext'}, axis=1) 
symptoms_group.head()

,patient_id,immediate_symptoms_resulting_ext
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,"Loss of Consciousness,Disorientation,Incoheren..."
1,00469456-99a5-4c99-aa48-a918986c7c45,"Loss of Consciousness,Confusion,Memory Loss,Da..."
2,012bfc75-70c1-4007-ab9e-1f4ee45bd537,"Loss of Consciousness,Disorientation,Incoheren..."
3,012eccde-f0b9-4ae2-aebf-3c821c461545,"Confusion,Memory Loss,Disorientation,Incoheren..."
4,01786d46-2829-42e2-8d50-135b7e212ea3,"Disorientation,Confusion,Extreme pain in the f..."


In [33]:
df_tbi_incident_ext = df_tbi_incident_ext.merge(symptoms_group, how='left', on='patient_id')
df_tbi_incident_ext.head(5)

,id,tbi_incident_date,describe_event,injury_from,head_hit_location,patient_id,total_tbi,immediate_symptoms_resulting,num_head_hit_location,injury_from_new,all_head_hit_locations,head_hit_location_new,immediate_symptoms_resulting_ext
0,1080,7/31/2013,I was riding a longboard to my friends house w...,Fall,Left Side of Head,efd7682b-adde-4b6f-b2d7-16f50597c984,1.0,"Loss of Consciousness,Disorientation,Incoheren...",NaN,Accident,"Left Side of Head,Left Side of Head",Left Side of Head,"Loss of Consciousness,Disorientation,Incoheren..."
1,1164,4/3/2024,"On april 3, i was preforming my work duties . ...",Accident,Left Side of Head,bb8e0677-ddc8-49dd-be08-ab6f10ad43f3,1.0,"Disorientation,Confusion,Dazed or Vacant Stare",NaN,Assault,Left Side of Head,Left Side of Head,"Disorientation,Confusion,Dazed or Vacant Stare"
2,1105,5/23/2023,Drank a bottle of window cleaner with amonia.,Collision,"Front of Head,Back of Head",4471257f-7b52-4278-9654-e56cab633e67,5.0,"Loss of Consciousness,Disorientation,Confusion...",NaN,Accident,"Front of Head,Back of Head","Front of Head,Back of Head","Loss of Consciousness,Disorientation,Confusion..."
3,1051,6/26/2022,I was tubing on a lake behind a boat and got t...,Accident,"Front of Head,Right Side of Head,Back of Head,...",cfba5f4c-8266-4e4c-bcf8-36e423ab0e35,2.0,"Disorientation,Confusion,Dazed or Vacant Stare",NaN,Accident,"Front of Head,Back of Head,Left Side of Head,R...","Front of Head,Right Side of Head,Back of Head,...","Disorientation,Confusion,Memory Loss,Dazed or ..."
4,1121,7/11/2023,Car Accident,Accident,Neck,fd6392ce-544f-4248-bf6a-9d9e454c8dfd,3.0,"Memory Loss,Sound sensitivity,Confusion,Disori...",NaN,Accident,Neck,Neck,"Memory Loss,Sound sensitivity,Confusion,Disori..."


In [34]:
df_tbi_incident_ext['immediate_symptoms_resulting_new'] = np.where(df_tbi_incident_ext['immediate_symptoms_resulting'].isna(), df_tbi_incident_ext['immediate_symptoms_resulting_ext'], df_tbi_incident_ext['immediate_symptoms_resulting'])

In [35]:
df_tbi_incident['immediate_symptoms_resulting'] = df_tbi_incident_ext['immediate_symptoms_resulting_new']

In [36]:
df_tbi_incident['head_hit_location'] = df_tbi_incident_ext['head_hit_location_new'] 

In [37]:
df_tbi_incident.columns

Index(['id', 'tbi_incident_date', 'describe_event', 'injury_from',
       'head_hit_location', 'patient_id', 'total_tbi',
       'immediate_symptoms_resulting', 'num_head_hit_location',
       'injury_from_new'],
      dtype='object')

In [38]:
df_tbi_incident["head_hit_location"] = df_tbi_incident["head_hit_location"].astype(str).str.strip().str.title()

In [39]:
hit_loc=df_tbi_incident['head_hit_location'].astype(str).str.strip().str.split(',', expand=True)

In [40]:
hit_locations = pd.concat([hit_loc[0], hit_loc[2], hit_loc[3], hit_loc[4], hit_loc[5], hit_loc[6]])
freq_trauma = pd.DataFrame(hit_locations.value_counts(), columns=["count"])

In [41]:
freq_trauma.head()

,count
Back Of Head,259
Front Of Head,227
Left Side Of Head,177
Right Side Of Head,147
Nan,97


In [42]:
#Values to look out should be over 5
list_trauma_loc = freq_trauma[freq_trauma['count']>5].index.tolist()
# list_trauma_loc[:20]
len(list_trauma_loc)

17

In [43]:
# Clean entries
trauma_cleaned_list = set([
    entry.strip().title() for entry in list_trauma_loc
    if entry.strip().title() not in ['Nan', 'No', 'Other']
])


In [44]:
replace_dict = {
    'Back': 'Back Of Head'
}

trauma_cleaned_list = set([replace_dict.get(val, val) for val in trauma_cleaned_list])
trauma_cleaned_list

{'All',
 'Back Of Head',
 'Front Of Head',
 'Left Side Of Head',
 'Neck',
 'Not Sure',
 'Right Side Of Head',
 'Top Of Head',
 'Whiplash'}

In [45]:
for i in trauma_cleaned_list:
    #df_tbi_incident['headhit_'+i.replace(" ", "_")] =  df_tbi_incident['head_hit_location'].str.findall(i)
    df_tbi_incident['headhit_'+i.replace(" ", "_")] = np.where(df_tbi_incident['head_hit_location'].str.find(i) >= 0, 1, 0)

In [54]:
df_tbi_incident['headhit_Right_Side_Of_Head'].sum()

np.int64(230)

In [46]:
df_tbi_incident.head(2)

,id,tbi_incident_date,describe_event,injury_from,head_hit_location,patient_id,total_tbi,immediate_symptoms_resulting,num_head_hit_location,injury_from_new,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,headhit_Right_Side_Of_Head
0,1080,7/31/2013,I was riding a longboard to my friends house w...,Fall,Left Side Of Head,efd7682b-adde-4b6f-b2d7-16f50597c984,1.0,"Loss of Consciousness,Disorientation,Incoheren...",NaN,Accident,0,0,0,1,0,0,0,0,0
1,1164,4/3/2024,"On april 3, i was preforming my work duties . ...",Accident,Left Side Of Head,bb8e0677-ddc8-49dd-be08-ab6f10ad43f3,1.0,"Disorientation,Confusion,Dazed or Vacant Stare",NaN,Assault,0,0,0,1,0,0,0,0,0


In [55]:
imm_symp=df_tbi_incident['immediate_symptoms_resulting'].str.split(',', expand=True)

In [56]:
immediate_symptoms = pd.concat([imm_symp[0], imm_symp[2], imm_symp[3], imm_symp[4], imm_symp[5], imm_symp[6]])
freq_symp = pd.DataFrame(immediate_symptoms.value_counts(), columns=["count"])
freq_symp.head(15)

,count
Confusion,463
Loss of Consciousness,306
Dazed or Vacant Stare,271
Memory Loss,256
Disorientation,226
Memory Loss,166
Incoherent Speech,146
Disorientation,126
Loss of Consciousness,117
Incoherent Speech,101


In [57]:
freq_symp.shape

(211, 1)

In [58]:
#Values to look out should be over 3
list_im_symp = freq_symp[freq_symp['count']>3].index.tolist()
len(list_im_symp)

21

In [59]:
list_im_symp

['Confusion',
 'Loss of Consciousness',
 'Dazed or Vacant Stare',
 'Memory Loss',
 'Disorientation',
 ' Memory Loss',
 'Incoherent Speech',
 ' Disorientation',
 ' Loss of Consciousness',
 ' Incoherent Speech',
 'nan',
 ' Dazed or Vacant Stare',
 'Headache',
 '',
 'Coma',
 'Dizziness',
 'Nausea',
 'Light sensitivity',
 ' Confusion',
 'Headaches',
 ' Headache']

In [60]:
# Step 1: Strip and title-case
cleaned_symptoms = [s.strip().title() for s in list_im_symp if s.strip()]

# Step 2: Remove invalid or ambiguous entries
invalid_values = ['Nan', '']  # Already excluded '' but keeping for clarity
cleaned_symptoms = [s for s in cleaned_symptoms if s not in invalid_values]

# Step 3: Normalize variants (e.g., "Headaches" → "Headache")
normalization_dict = {
    "Headaches": "Headache",
    
}

cleaned_symptoms = set([
    normalization_dict.get(symptom, symptom) for symptom in cleaned_symptoms
])

cleaned_symptoms = [s.lower() for s in cleaned_symptoms]
cleaned_symptoms

['light sensitivity',
 'headache',
 'dazed or vacant stare',
 'dizziness',
 'disorientation',
 'nausea',
 'confusion',
 'coma',
 'incoherent speech',
 'memory loss',
 'loss of consciousness']

In [61]:
for i in cleaned_symptoms:
    #df_tbi_incident['headhit_'+i.replace(" ", "_")] =  df_tbi_incident['head_hit_location'].str.findall(i)
    df_tbi_incident['immediate_symptoms_resulting'] = df_tbi_incident['immediate_symptoms_resulting'].astype(str).fillna('').str.strip().str.lower()
    df_tbi_incident['imm_symp_'+i.replace(" ", "_")] = np.where(df_tbi_incident['immediate_symptoms_resulting'].str.find(i) >= 0, 1, 0)

In [64]:
df_tbi_incident['imm_symp_loss_of_consciousness'].sum()

np.int64(448)

In [63]:
df_tbi_incident.head(2)

,id,tbi_incident_date,describe_event,injury_from,head_hit_location,patient_id,total_tbi,immediate_symptoms_resulting,num_head_hit_location,injury_from_new,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,headhit_Right_Side_Of_Head,imm_symp_light_sensitivity,imm_symp_headache,imm_symp_dazed_or_vacant_stare,imm_symp_dizziness,imm_symp_disorientation,imm_symp_nausea,imm_symp_confusion,imm_symp_coma,imm_symp_incoherent_speech,imm_symp_memory_loss,imm_symp_loss_of_consciousness
0,1080,7/31/2013,I was riding a longboard to my friends house w...,Fall,Left Side Of Head,efd7682b-adde-4b6f-b2d7-16f50597c984,1.0,"loss of consciousness,disorientation,incoheren...",NaN,Accident,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,1,1,1
1,1164,4/3/2024,"On april 3, i was preforming my work duties . ...",Accident,Left Side Of Head,bb8e0677-ddc8-49dd-be08-ab6f10ad43f3,1.0,"disorientation,confusion,dazed or vacant stare",NaN,Assault,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0


In [65]:
list_car = [ "car", "Car", "cars", "Cars", "driving", "Driving", "drive", "Drive",
    "driver", "Driver", "driven", "drives", "vehicle", "Vehicle",
    "automobile", "Automobile", "sedan", "Sedan", "convertible", "Convertible",
    "hatchback", "Hatchback", "bumper", "Bumper", "windshield", "Windshield",
    "ran over", "run over", "hit by car", "hit-and-run", "road", "roadway","crash", "Crash", "auto", "Auto", "SUV", "suv",
    "pickup", "Pickup", "minivan", "Minivan", "parking lot",]
df_tbi_incident['event_desc_car'] = np.where(df_tbi_incident['describe_event'].str.contains('|'.join(list_car)) >0, 1, 0)

In [66]:
list_fall = ['fall', 'fell', 'Fall', 'Fell',  "tumble", "Tumble", "tumbled", "Tumbled", "stumble", "Stumble",
    "stumbled", "Stumbled", "topple", "Topple", "toppled", "Toppled",
    "collapse", "Collapse", "collapsed", "Collapsed", "slipped", "Slipped",]
df_tbi_incident['event_desc_fall'] = np.where(df_tbi_incident['describe_event'].str.contains('|'.join(list_fall)) >0, 1, 0)

In [67]:
list_severe = ['hospital', 'coma']
df_tbi_incident['event_desc_severe'] = np.where(df_tbi_incident['describe_event'].str.contains('|'.join(list_severe)) >0, 1, 0)

In [68]:
freq_origin = pd.DataFrame(df_tbi_incident['injury_from'].value_counts())
freq_origin.head(15)

,count
injury_from,
Accident,383
Fall,158
Collision,103
Sports,81
Assault,77
Stroke,11
Surgery,6
Covid illness,3
Neurosurgery,2


In [69]:
freq_origin.columns = ['count']

In [70]:
#Values to look out should be over or equal to 5
list_origin = freq_origin[freq_origin['count']>=5].index.tolist()
list_origin[:10]

['Accident', 'Fall', 'Collision', 'Sports', 'Assault', 'Stroke', 'Surgery']

In [71]:
len(list_origin)

7

In [72]:
for i in list_origin:
    #df_tbi_incident['headhit_'+i.replace(" ", "_")] =  df_tbi_incident['head_hit_location'].str.findall(i)
    df_tbi_incident['injury_from_'+i.replace(" ", "_")] = np.where(df_tbi_incident['injury_from'].str.find(i) >= 0, 1, 0)

In [73]:
df_tbi_incident_pl = df_tbi_incident.groupby(['patient_id']).agg(

    first_tbi_date=('tbi_incident_date', 'min'),

    last_tbi_date=('tbi_incident_date', 'max'),
    
    first_tbi_from=('injury_from', 'first'), 
    
    first_tbi_desc=('describe_event', 'first')
    )
df_tbi_incident_pl = df_tbi_incident_pl.reset_index()


In [74]:
df_tbi_incident_pl.head(5)

,patient_id,first_tbi_date,last_tbi_date,first_tbi_from,first_tbi_desc
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,7/2/1999,7/2/1999,Accident,"Automobile accident, ran into cars and flipped..."
1,00469456-99a5-4c99-aa48-a918986c7c45,2/9/2021,2/9/2021,Subarachnoid haemorrhage,I collapsed at work showing stroke like sympto...
2,012bfc75-70c1-4007-ab9e-1f4ee45bd537,8/12/2014,8/12/2014,Accident,Hit by a car checking my mail. The car hit me ...
3,012eccde-f0b9-4ae2-aebf-3c821c461545,10/14/2022,10/21/2022,Fall,I fell down stairs
4,01786d46-2829-42e2-8d50-135b7e212ea3,3/1/2019,3/1/2019,Accident,Not sure which day in March. I went to sit bac...


In [75]:
sub_set = df_tbi_incident[list(df_tbi_incident.columns)[5:]]
result = sub_set.groupby(by=['patient_id']).sum() 
result = result.reset_index()

In [77]:
result['injury_from_Accident'].sum()

np.int64(384)

In [ ]:
result.head(2)

,patient_id,total_tbi,immediate_symptoms_resulting,num_head_hit_location,injury_from_new,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,headhit_Right_Side_Of_Head,imm_symp_light_sensitivity,imm_symp_headache,imm_symp_dazed_or_vacant_stare,imm_symp_dizziness,imm_symp_disorientation,imm_symp_nausea,imm_symp_confusion,imm_symp_coma,imm_symp_incoherent_speech,imm_symp_memory_loss,imm_symp_loss_of_consciousness,event_desc_car,event_desc_fall,event_desc_severe,injury_from_Accident,injury_from_Fall,injury_from_Collision,injury_from_Sports,injury_from_Assault,injury_from_Stroke,injury_from_Surgery
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1.0,loss of consciousness,1,Colision,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,0,0,0,0
1,00469456-99a5-4c99-aa48-a918986c7c45,1.0,"loss of consciousness,incoherent speech,memory...",1,Fall,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0


In [78]:
final_tbi_incident = df_tbi_incident_pl.merge(result, how='left', on='patient_id')
final_tbi_incident.shape

(911, 39)

In [79]:
final_tbi_incident.head(2)

,patient_id,first_tbi_date,last_tbi_date,first_tbi_from,first_tbi_desc,total_tbi,immediate_symptoms_resulting,num_head_hit_location,injury_from_new,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,headhit_Right_Side_Of_Head,imm_symp_light_sensitivity,imm_symp_headache,imm_symp_dazed_or_vacant_stare,imm_symp_dizziness,imm_symp_disorientation,imm_symp_nausea,imm_symp_confusion,imm_symp_coma,imm_symp_incoherent_speech,imm_symp_memory_loss,imm_symp_loss_of_consciousness,event_desc_car,event_desc_fall,event_desc_severe,injury_from_Accident,injury_from_Fall,injury_from_Collision,injury_from_Sports,injury_from_Assault,injury_from_Stroke,injury_from_Surgery
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,7/2/1999,7/2/1999,Accident,"Automobile accident, ran into cars and flipped...",1.0,loss of consciousness,1,Colision,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,0,0,0,0
1,00469456-99a5-4c99-aa48-a918986c7c45,2/9/2021,2/9/2021,Subarachnoid haemorrhage,I collapsed at work showing stroke like sympto...,1.0,"loss of consciousness,incoherent speech,memory...",1,Fall,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0


In [80]:
# final_tbi_incident.isna().sum()

# New Resulting Factors

In [81]:
# Load the data
new_resulting_factors = pd.read_csv('../data/raw/new_resulting_factors.csv')
new_resulting_factors.shape

C:\Users\rohit\AppData\Local\Temp\ipykernel_7656\2254881688.py:2: DtypeWarning: Columns (0,4,12) have mixed types. Specify dtype option on import or set low_memory=False.
  new_resulting_factors = pd.read_csv('../data/raw/new_resulting_factors.csv')


(189782, 13)

In [82]:
print((new_resulting_factors.factor.unique()))
print(len(new_resulting_factors.patient_about_id.unique()))

['Loud Noises' 'Fatigue' 'Low Temperature' ... 'Stress eating'
 "Crohn's flare" 'Rain during drive']
1049


In [83]:
# new_resulting_factors.head(2)
new_resulting_factors = new_resulting_factors.iloc[:,:12]

In [84]:
new_resulting_factors['symptom_id'].astype(str)
new_resulting_factors['symptom_id'] = pd.to_numeric(new_resulting_factors['symptom_id'], errors='coerce')
new_resulting_factors = new_resulting_factors.dropna(subset=['symptom_id'])
new_resulting_factors.shape

(189773, 12)

In [85]:
subcategories = {'sleep','emotional','speech','cognitive','vision','physical'}

# drop symptom_date == Null
new_resulting_factors = new_resulting_factors.dropna(subset=['symptom_date'])

#keeping main categories to use
new_resulting_factors = new_resulting_factors[new_resulting_factors['subcategory'].isin(subcategories)]
print(len(new_resulting_factors.patient_about_id.unique()))

# group minor symptoms <5 as symptom_id = 9999
new_resulting_factors.loc[new_resulting_factors['symptom_id'] < 5, 'symptom_id'] = 9999

1023


In [86]:
# frequency of the top 100 symptoms by patient
new_resulting_factors['subcategory_count'] = new_resulting_factors.groupby(['patient_about_id','subcategory'])['subcategory'].transform('count')
#new_resulting_factors_100_symotoms['symptom_count'] = new_resulting_factors_100_symotoms.groupby(['patient_about_id','subcategory'])['subcategory'].transform('count')
print(new_resulting_factors.shape)
new_resulting_factors.head()
print(len(new_resulting_factors.patient_about_id.unique()))

(90196, 13)
1023


In [87]:
# new_resulting_factors.dtypes
new_resulting_factors['severity'] = pd.to_numeric(new_resulting_factors['severity'], errors='coerce')

In [88]:
# min and max severity for top 100 symptoms by patient
new_resulting_factors['severity_max'] = new_resulting_factors.groupby(['patient_about_id','subcategory'])['severity'].transform('max')
new_resulting_factors['severity_min'] = new_resulting_factors.groupby(['patient_about_id','subcategory'])['severity'].transform('min')


In [89]:
grouped = new_resulting_factors.groupby(['patient_about_id', 'subcategory'])['severity']
new_resulting_factors['severity_mean'] = grouped.transform('mean')
new_resulting_factors['severity_std'] = grouped.transform('std')       # Standard deviation
new_resulting_factors['severity_median'] = grouped.transform('median')
new_resulting_factors['severity_var'] = grouped.transform('var')       # Variance

In [90]:
# date of first and last symptom for top 100 symptoms by patient
new_resulting_factors['first_symptom_date'] = new_resulting_factors.groupby(['patient_about_id','subcategory'])['symptom_date'].transform('min')
new_resulting_factors['last_symptom_date'] = new_resulting_factors.groupby(['patient_about_id','subcategory'])['symptom_date'].transform('max')
print(new_resulting_factors.shape)
new_resulting_factors.head()
print(len(new_resulting_factors.patient_about_id.unique()))

(90196, 21)
1023


In [91]:
# date of maximum severity per symptom and patient
indices_max = new_resulting_factors.groupby(['patient_about_id','subcategory'])['severity'].transform('max') == new_resulting_factors['severity_max']
max_date = new_resulting_factors[indices_max][['patient_about_id','subcategory','symptom_date']]
max_date = max_date.rename(columns={"symptom_date": "max_severity_date"})

In [92]:
# remove duplicates from the original dataframe before merging
new_resulting_factors = new_resulting_factors.drop_duplicates(subset=['patient_about_id','subcategory'])
new_resulting_factors.shape

(4554, 21)

In [93]:
# Merge date of maximum severity to the original dataframe
new_resulting_factors = pd.merge(new_resulting_factors , max_date, how = 'outer', on=['patient_about_id','subcategory'])
print(new_resulting_factors.shape)
new_resulting_factors.head()
print(len(new_resulting_factors.patient_about_id.unique()))

(79678, 22)
1023


In [94]:
# date of minimum severity per symptom and patient
indices_min = new_resulting_factors.groupby(['patient_about_id','subcategory'])['severity'].transform('min') == new_resulting_factors['severity_min']
min_date = new_resulting_factors[indices_min][['patient_about_id','subcategory','symptom_date']]
min_date = min_date.rename(columns={"symptom_date": "min_severity_date"})

In [95]:
# remove duplicates from the original dataframe before merging
new_resulting_factors = new_resulting_factors.drop_duplicates(subset=['patient_about_id','subcategory'])

# Merge date of minimum severity to the original dataframe
new_resulting_factors = pd.merge(new_resulting_factors , min_date, how = 'outer', on=['patient_about_id','subcategory'])
print(new_resulting_factors.shape)
print(len(new_resulting_factors.patient_about_id.unique()))

(6785, 23)
1023


In [96]:
new_resulting_factors.columns

Index(['id', 'user_input_id', 'symptom_date', 'had_symptom', 'severity',
       'description', 'symptom_id', 'factor', 'category', 'subcategory',
       'logged_at', 'patient_about_id', 'subcategory_count', 'severity_max',
       'severity_min', 'severity_mean', 'severity_std', 'severity_median',
       'severity_var', 'first_symptom_date', 'last_symptom_date',
       'max_severity_date', 'min_severity_date'],
      dtype='object')

In [97]:
# final data frame for new_resulting_factors
new_resulting_factors = new_resulting_factors[['patient_about_id',
                                               'subcategory',
                                               'severity',
                                               'subcategory_count',
                                               'severity_max',
                                               'severity_min',
                                               'severity_mean', 
                                               'severity_std', 
                                               'severity_median',
                                               'severity_var',
                                               'first_symptom_date',
                                               'last_symptom_date',
                                               'max_severity_date',
                                               'min_severity_date']]

print(new_resulting_factors.shape)

(6785, 14)


In [98]:
# find date difference between first subcategory to subcategory with highest severity level
from datetime import datetime
new_resulting_factors['first_symptom_date'] = pd.to_datetime(new_resulting_factors['first_symptom_date'],format='%Y-%m-%d',errors='coerce')
new_resulting_factors['max_severity_date'] = pd.to_datetime(new_resulting_factors['max_severity_date'],format='%Y-%m-%d',errors='coerce')

Start = new_resulting_factors.first_symptom_date
End  = new_resulting_factors.max_severity_date
new_resulting_factors['day_of_max_severity'] = End.subtract(Start).dt.days

In [99]:
print(new_resulting_factors.shape)
new_resulting_factors.head()
print(len(new_resulting_factors.patient_about_id.unique()))

df = new_resulting_factors
df_1 = df.groupby('patient_about_id').agg(lambda x: x.tolist())
final_df =  df.groupby('patient_about_id').agg(lambda x: x.tolist())
print(final_df.shape)
final_df.head()

(6785, 15)
1023
(1023, 14)


,subcategory,severity,subcategory_count,severity_max,severity_min,severity_mean,severity_std,severity_median,severity_var,first_symptom_date,last_symptom_date,max_severity_date,min_severity_date,day_of_max_severity
patient_about_id,,,,,,,,,,,,,,
0006ad41-c2d3-4994-8aab-7a3a107d50aa,"[cognitive, emotional, physical, sleep, speech...","[nan, nan, nan, nan, nan, nan]","[7, 10, 11, 7, 14, 6]","[nan, 10.0, nan, nan, nan, nan]","[nan, 10.0, nan, nan, nan, nan]","[nan, 10.0, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan]","[nan, 10.0, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan]","[NaT, NaT, NaT, NaT, NaT, NaT]","[7/1/1999 0:00, 7/1/1999 0:00, 7/1/1999 0:00, ...","[NaT, NaT, NaT, NaT, NaT, NaT]","[nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan]"
00469456-99a5-4c99-aa48-a918986c7c45,"[cognitive, emotional, emotional, emotional, e...","[73.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[8, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4, 4, 2]","[73.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[72.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[72.5, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[0.7071067811865476, nan, nan, nan, nan, nan, ...","[72.5, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[0.5, nan, nan, nan, nan, nan, nan, nan, nan, ...","[NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, ...","[5/20/2021 0:00, 5/20/2021 0:00, 5/20/2021 0:0...","[NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, ...","[nan, 5/20/2021 0:00, 5/20/2021 0:00, 5/20/202...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
012bfc75-70c1-4007-ab9e-1f4ee45bd537,"[cognitive, emotional, physical, sleep, vision]","[nan, nan, nan, nan, nan]","[6, 11, 14, 8, 2]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[NaT, NaT, NaT, NaT, NaT]","[8/12/2014 0:00, 8/12/2014 0:00, 8/12/2014 0:0...","[NaT, NaT, NaT, NaT, NaT]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]"
012eccde-f0b9-4ae2-aebf-3c821c461545,"[cognitive, emotional, physical, sleep, speech...","[nan, nan, nan, nan, nan, nan]","[28, 7, 13, 2, 15, 8]","[81.0, 100.0, nan, nan, nan, nan]","[49.0, 0.0, nan, nan, nan, nan]","[73.0, 45.333333333333336, nan, nan, nan, nan]","[11.590225767142474, 50.64911976859354, nan, n...","[78.0, 36.0, nan, nan, nan, nan]","[134.33333333333334, 2565.3333333333335, nan, ...","[NaT, NaT, NaT, NaT, NaT, NaT]","[8/7/2023 0:00, 8/7/2023 0:00, 10/13/2022 0:00...","[NaT, NaT, NaT, NaT, NaT, NaT]","[nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan]"
01786d46-2829-42e2-8d50-135b7e212ea3,"[cognitive, emotional, physical, speech, vision]","[nan, nan, nan, nan, nan]","[2, 1, 4, 2, 3]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[NaT, NaT, NaT, NaT, NaT]","[2/28/2019 0:00, 2/28/2019 0:00, 2/28/2019 0:0...","[NaT, NaT, NaT, NaT, NaT]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]"


In [100]:

subcategories = list(df.subcategory.unique())

for cat in subcategories:
    count = []
    sev_max = []
    sev_min = []
    sev_mean = []
    sev_std = []
    sev_median = []
    sev_var = []

    first_date = []
    min_date = []
    last_date = []
    max_date = []

    for index, row in df_1.iterrows():
        cat_list = row['subcategory']

        if cat in cat_list:
            factor_index = cat_list.index(cat)
            count.append(row['subcategory_count'][factor_index])
            sev_max.append(row['severity_max'][factor_index])
            sev_min.append(row['severity_min'][factor_index])
            sev_mean.append(row['severity_mean'][factor_index])
            sev_std.append(row['severity_std'][factor_index])
            sev_median.append(row['severity_median'][factor_index])
            sev_var.append(row['severity_var'][factor_index])

            first_date.append(row['first_symptom_date'][factor_index])
            min_date.append(row['min_severity_date'][factor_index])
            last_date.append(row['last_symptom_date'][factor_index])
            max_date.append(row['max_severity_date'][factor_index])
        else:
            count.append(np.nan)
            sev_max.append(np.nan)
            sev_min.append(np.nan)
            sev_mean.append(np.nan)
            sev_std.append(np.nan)
            sev_median.append(np.nan)
            sev_var.append(np.nan)

            first_date.append(np.nan)
            min_date.append(np.nan)
            last_date.append(np.nan)
            max_date.append(np.nan)

    final_df[f'subcategory_count_{cat}'] = count
    final_df[f'severity_max_{cat}'] = sev_max
    final_df[f'severity_min_{cat}'] = sev_min
    final_df[f'severity_mean_{cat}'] = sev_mean
    final_df[f'severity_std_{cat}'] = sev_std
    final_df[f'severity_median_{cat}'] = sev_median
    final_df[f'severity_var_{cat}'] = sev_var

    final_df[f'first_date_{cat}'] = first_date
    final_df[f'min_date_{cat}'] = min_date
    final_df[f'max_date_{cat}'] = max_date
    final_df[f'last_date_{cat}'] = last_date


In [101]:
# extract top 100 most frequent symptoms
#n = 100
#symptom_100_list = new_resulting_factors['symptom_id'].value_counts()[:n].index.tolist()

# keep observations with only top 100 symptoms
#new_resulting_factors_100_symotoms = new_resulting_factors[new_resulting_factors.symptom_id.isin(symptom_100_list)]
#new_resulting_factors.head()


# # date of minimum severity per symptom and patient
# indices_min = new_resulting_factors.groupby(['patient_about_id','subcategory'])['severity'].transform('min') == new_resulting_factors['severity_min']
# min_date = new_resulting_factors[indices_min][['patient_about_id','subcategory','symptom_date']]
# min_date = min_date.rename(columns={"symptom_date": "min_severity_date"})

# # remove duplicates from the original dataframe before merging
# new_resulting_factors = new_resulting_factors.drop_duplicates(subset=['patient_about_id','subcategory'])

# Merge date of minimum severity to the original dataframe
# new_resulting_factors = pd.merge(new_resulting_factors , min_date, how = 'outer', on=['patient_about_id','subcategory'])
# print(new_resulting_factors.shape)
# print(len(new_resulting_factors.patient_about_id.unique()))

# remove duplicates
#new_resulting_factors = new_resulting_factors.drop_duplicates(subset=['patient_about_id','subcategory'])
# print(new_resulting_factors.shape)
# # final data frame for new_resulting_factors
# new_resulting_factors = new_resulting_factors[['patient_about_id',
#                                                'subcategory',
#                                                'severity',
#                                                'subcategory_count',
#                                                'severity_max',
#                                                'severity_min',
#                                                'first_symptom_date',
#                                                'last_symptom_date',
#                                                'max_severity_date',
#                                                'min_severity_date']]

# find date difference between first subcategory to subcategory with highest severity level
# from datetime import datetime
# new_resulting_factors['first_symptom_date'] = pd.to_datetime(new_resulting_factors['first_symptom_date'],format='%Y-%m-%d',errors='coerce')
# new_resulting_factors['max_severity_date'] = pd.to_datetime(new_resulting_factors['max_severity_date'],format='%Y-%m-%d',errors='coerce')

# Start = new_resulting_factors.first_symptom_date
# End  = new_resulting_factors.max_severity_date
# new_resulting_factors['day_of_max_severity'] = End.subtract(Start).dt.days


# print(new_resulting_factors.shape)
# new_resulting_factors.head()
# print(len(new_resulting_factors.patient_about_id.unique()))

# df = new_resulting_factors
# df_1 = df.groupby('patient_about_id').agg(lambda x: x.tolist())
# final_df =  df.groupby('patient_about_id').agg(lambda x: x.tolist())
# print(final_df.shape)
# final_df.head()

# subcategories = list(df.subcategory.unique())
# df_1.head()

# for cat in subcategories:
#     count = []
#     sev_max = []
#     sev_min = []
#     first_date = []
#     min_date = []
#     last_date = []
#     max_date = []
#     for index, row in df_1.iterrows():
#         cat_list = (row['subcategory'])
#         if cat in cat_list:
#             factor_index = cat_list.index(cat)
#             count.append(row['subcategory_count'][factor_index])
#             sev_max.append(row['severity_max'][factor_index])
#             sev_min.append(row['severity_min'][factor_index])
#             first_date.append(row['first_symptom_date'][factor_index])
#             min_date.append(row['min_severity_date'][factor_index])
#             last_date.append(row['last_symptom_date'][factor_index])
#             max_date.append(row['max_severity_date'][factor_index])  
#         else:
#             count.append(np.nan)
#             sev_max.append(np.nan)
#             sev_min.append(np.nan)
#             first_date.append(np.nan)
#             min_date.append(np.nan)
#             last_date.append(np.nan)
#             max_date.append(np.nan)  
#     final_df['subcategory_count_{}'.format(cat)] = count
#     final_df['max_date_{}'.format(cat)] = max_date
#     final_df['min_date_{}'.format(cat)] = min_date
#     final_df['first_date_{}'.format(cat)] = first_date
#     final_df['last_date_{}'.format(cat)] = last_date
#     final_df['severity_max_{}'.format(cat)] = sev_max
#     final_df['severity_min_{}'.format(cat)] = sev_max

In [102]:
final_df.shape

(1023, 80)

In [103]:
# final_df.isna().sum()

In [104]:
final_df.head(2)

,subcategory,severity,subcategory_count,severity_max,severity_min,severity_mean,severity_std,severity_median,severity_var,first_symptom_date,last_symptom_date,max_severity_date,min_severity_date,day_of_max_severity,subcategory_count_cognitive,severity_max_cognitive,severity_min_cognitive,severity_mean_cognitive,severity_std_cognitive,severity_median_cognitive,severity_var_cognitive,first_date_cognitive,min_date_cognitive,max_date_cognitive,last_date_cognitive,subcategory_count_emotional,severity_max_emotional,severity_min_emotional,severity_mean_emotional,severity_std_emotional,severity_median_emotional,severity_var_emotional,first_date_emotional,min_date_emotional,max_date_emotional,last_date_emotional,subcategory_count_physical,severity_max_physical,severity_min_physical,severity_mean_physical,severity_std_physical,severity_median_physical,severity_var_physical,first_date_physical,min_date_physical,max_date_physical,last_date_physical,subcategory_count_sleep,severity_max_sleep,severity_min_sleep,severity_mean_sleep,severity_std_sleep,severity_median_sleep,severity_var_sleep,first_date_sleep,min_date_sleep,max_date_sleep,last_date_sleep,subcategory_count_speech,severity_max_speech,severity_min_speech,severity_mean_speech,severity_std_speech,severity_median_speech,severity_var_speech,first_date_speech,min_date_speech,max_date_speech,last_date_speech,subcategory_count_vision,severity_max_vision,severity_min_vision,severity_mean_vision,severity_std_vision,severity_median_vision,severity_var_vision,first_date_vision,min_date_vision,max_date_vision,last_date_vision
patient_about_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0006ad41-c2d3-4994-8aab-7a3a107d50aa,"[cognitive, emotional, physical, sleep, speech...","[nan, nan, nan, nan, nan, nan]","[7, 10, 11, 7, 14, 6]","[nan, 10.0, nan, nan, nan, nan]","[nan, 10.0, nan, nan, nan, nan]","[nan, 10.0, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan]","[nan, 10.0, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan]","[NaT, NaT, NaT, NaT, NaT, NaT]","[7/1/1999 0:00, 7/1/1999 0:00, 7/1/1999 0:00, ...","[NaT, NaT, NaT, NaT, NaT, NaT]","[nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan]",7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,7/1/1999 0:00,10.0,10.0,10.0,10.0,NaN,10.0,NaN,NaT,NaN,NaT,7/1/1999 0:00,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,7/1/1999 0:00,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,7/1/1999 0:00,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,7/1/1999 0:00,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,7/1/1999 0:00
00469456-99a5-4c99-aa48-a918986c7c45,"[cognitive, emotional, emotional, emotional, e...","[73.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[8, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4, 4, 2]","[73.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[72.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[72.5, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[0.7071067811865476, nan, nan, nan, nan, nan, ...","[72.5, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, nan...","[0.5, nan, nan, nan, nan, nan, nan, nan, nan, ...","[NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, ...","[5/20/2021 0:00, 5/20/2021 0:00, 5/20/2021 0:0...","[NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, ...","[nan, 5/20/2021 0:00, 5/20/2021 0:00, 5/20/202...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",8.0,73.0,72.0,72.5,0.707107,72.5,0.5,NaT,NaN,NaT,5/20/2021 0:00,6.0,50.0,50.0,50.0,NaN,50.0,NaN,NaT,5/20/2021 0:00,NaT,5/20/2021 0:00,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,2/9/2021 0:00,4.0,42.0,42.0,42.0,NaN,42.0,NaN,NaT,5/20/2021 0:00,NaT,5/20/2021 0:00,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,2/9/2021 0:00,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,2/9/2021 0:00


In [105]:
# Load the data
# new_resulting_factors = pd.read_csv('data/second_batch/new_resulting_factors.csv')

In [106]:
# drop symptom_date == Null
# new_resulting_factors = new_resulting_factors.dropna(subset=['symptom_date'])

In [107]:
# # group minor symptoms <5 as symptom_id = 9999
# new_resulting_factors.loc[new_resulting_factors['symptom_id'] < 5, 'symptom_id'] = 9999

In [108]:
# # extract top 100 most frequent symptoms
# n = 100
# symptom_100_list = new_resulting_factors['symptom_id'].value_counts()[:n].index.tolist()

In [109]:
# # keep observations with only top 100 symptoms
# new_resulting_factors_100_symotoms = new_resulting_factors[new_resulting_factors.symptom_id.isin(symptom_100_list)]

In [110]:
# # frequency of the top 100 symptoms by patient
# new_resulting_factors_100_symotoms['symptom_count'] = new_resulting_factors_100_symotoms.groupby(['patient_about_id','symptom_id'])['symptom_id'].transform('count')

In [111]:
# # min and max severity for top 100 symptoms by patient
# new_resulting_factors_100_symotoms['severity_max'] = new_resulting_factors_100_symotoms.groupby(['patient_about_id','symptom_id'])['severity'].transform('max')
# new_resulting_factors_100_symotoms['severity_min'] = new_resulting_factors_100_symotoms.groupby(['patient_about_id','symptom_id'])['severity'].transform('min')

In [112]:
# # date of first and last symptom for top 100 symptoms by patient
# new_resulting_factors_100_symotoms['first_symptom_date'] = new_resulting_factors_100_symotoms.groupby(['patient_about_id','symptom_id'])['symptom_date'].transform('min')
# new_resulting_factors_100_symotoms['last_symptom_date'] = new_resulting_factors_100_symotoms.groupby(['patient_about_id','symptom_id'])['symptom_date'].transform('max')

In [113]:
# # date of maximum severity per symptom and patient
# indices_max = new_resulting_factors_100_symotoms.groupby(['patient_about_id','symptom_id'])['severity'].transform('max') == new_resulting_factors_100_symotoms['severity_max']
# max_date = new_resulting_factors_100_symotoms[indices_max][['patient_about_id','symptom_id','symptom_date']]
# max_date = max_date.rename(columns={"symptom_date": "max_severity_date"})

# # remove duplicates from the original dataframe before merging
# new_resulting_factors_100_symotoms_nodups = new_resulting_factors_100_symotoms.drop_duplicates(subset=['patient_about_id','symptom_id'])

# # Merge date of maximum severity to the original dataframe
# new_resulting_factors_100_symotoms_merged = pd.merge(new_resulting_factors_100_symotoms_nodups , max_date, how = 'inner', on=['patient_about_id','symptom_id'])



In [114]:
# # date of minimum severity per symptom and patient
# indices_min = new_resulting_factors_100_symotoms.groupby(['patient_about_id','symptom_id'])['severity'].transform('min') == new_resulting_factors_100_symotoms['severity_min']
# min_date = new_resulting_factors_100_symotoms[indices_min][['patient_about_id','symptom_id','symptom_date']]
# min_date = min_date.rename(columns={"symptom_date": "min_severity_date"})

# # remove duplicates from the original dataframe before merging
# new_resulting_factors_100_symotoms_merged = new_resulting_factors_100_symotoms_merged.drop_duplicates(subset=['patient_about_id','symptom_id'])

# # Merge date of minimum severity to the original dataframe
# new_resulting_factors_100_symotoms_merged2 = pd.merge(new_resulting_factors_100_symotoms_merged , min_date, how = 'inner', on=['patient_about_id','symptom_id'])



In [115]:
# # remove duplicates
# new_resulting_factors_100_symotoms_long = new_resulting_factors_100_symotoms_merged2.drop_duplicates(subset=['patient_about_id','symptom_id'])

# # final data frame for new_resulting_factors
# new_resulting_factors_100_symotoms_long = new_resulting_factors_100_symotoms_long[['patient_about_id',
#                                                                                    'symptom_id',
#                                                                                    'factor',
#                                                                                    'subcategory',
#                                                                                    'severity',
#                                                                                    'symptom_count',
#                                                                                    'severity_max',
#                                                                                    'severity_min',
#                                                                                    'first_symptom_date',
#                                                                                    'last_symptom_date',
#                                                                                    'max_severity_date',
#                                                                                    'min_severity_date']]       

In [116]:
# df = new_resulting_factors_100_symotoms_long
# df_1 = df.groupby('patient_about_id').agg(lambda x: x.tolist())
# final_new_resulting_factors =  df.groupby('patient_about_id').agg(lambda x: x.tolist())

In [117]:
# factors = list(df.factor.unique())

In [118]:
# for factor in factors:
#     count = []
#     sev_max = []
#     sev_min = []
#     subcategory = []
#     first_date = []
#     min_date = []
#     last_date = []
#     max_date = []
#     factor_yes = []
#     for index, row in df_1.iterrows():
#         factor_list = (row['factor'])
#         if factor in factor_list:
#             factor_index = factor_list.index(factor)
#             factor_yes.append(1)
#             count.append(row['symptom_count'][factor_index])
#             sev_max.append(row['severity_max'][factor_index])
#             sev_min.append(row['severity_min'][factor_index])
#             subcategory.append(row['subcategory'][factor_index])
#             first_date.append(row['first_symptom_date'][factor_index])
#             min_date.append(row['symptom_count'][factor_index])
#             last_date.append(row['last_symptom_date'][factor_index])
#             max_date.append(row['symptom_count'][factor_index])  
#         else:
#             factor_yes.append(0)
#             count.append(np.nan)
#             sev_max.append(np.nan)
#             sev_min.append(np.nan)
#             subcategory.append(np.nan)
#             first_date.append(np.nan)
#             min_date.append(np.nan)
#             last_date.append(np.nan)
#             max_date.append(np.nan)  
#     final_new_resulting_factors['symptom_count_{}'.format(factor)] = count
#     final_new_resulting_factors['factor_{}'.format(factor)] = factor_yes
#     final_new_resulting_factors['max_date_{}'.format(factor)] = max_date
#     final_new_resulting_factors['min_date_{}'.format(factor)] = min_date
#     final_new_resulting_factors['first_date_{}'.format(factor)] = first_date
#     final_new_resulting_factors['last_date_{}'.format(factor)] = last_date
#     final_new_resulting_factors['subcategory_{}'.format(factor)] = subcategory
#     final_new_resulting_factors['severity_max_{}'.format(factor)] = sev_max
#     final_new_resulting_factors['severity_min_{}'.format(factor)] = sev_max


# Merge

In [119]:
final_new_resulting_factors = final_df

In [120]:
final_new_resulting_factors.shape

(1023, 80)

In [121]:
#turning resulting_factors into tuples
for item in list(final_new_resulting_factors.columns):
    if type(final_new_resulting_factors[item][0]) == list:
        final_new_resulting_factors[item] = getattr(final_new_resulting_factors, item).apply(tuple)

C:\Users\rohit\AppData\Local\Temp\ipykernel_7656\3670868669.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if type(final_new_resulting_factors[item][0]) == list:


In [122]:
#dropping duplicate column/index of patient_id
final_notes = final_notes[['note1', 'note1_score', 'note2', 'note2_score', 'note3',
       'note3_score', 'note1_post_date', 'note2_post_date', 'note3_post_date']]

In [123]:
patient_tbi =patient_info.merge(final_tbi_incident,how='left',on="patient_id")
patient_tbi_factors =patient_tbi.merge(final_new_resulting_factors,how='left',left_on="patient_id",right_index=True)
pofp_df = patient_tbi_factors.merge(final_notes,how='left',left_on="patient_id",right_index=True)

In [124]:
pofp_df.shape

(1240, 132)

In [125]:
pofp_df['patient_id'].nunique()

1240

In [126]:
# pofp_df.columns
column_list = pofp_df.columns.tolist()
print(column_list)

['patient_id', 'date_of_birth', 'gender', 'patient_type', 'external_id', 'first_tbi_date', 'last_tbi_date', 'first_tbi_from', 'first_tbi_desc', 'total_tbi', 'immediate_symptoms_resulting', 'num_head_hit_location', 'injury_from_new', 'headhit_Not_Sure', 'headhit_Neck', 'headhit_Top_Of_Head', 'headhit_Left_Side_Of_Head', 'headhit_Front_Of_Head', 'headhit_Back_Of_Head', 'headhit_All', 'headhit_Whiplash', 'headhit_Right_Side_Of_Head', 'imm_symp_light_sensitivity', 'imm_symp_headache', 'imm_symp_dazed_or_vacant_stare', 'imm_symp_dizziness', 'imm_symp_disorientation', 'imm_symp_nausea', 'imm_symp_confusion', 'imm_symp_coma', 'imm_symp_incoherent_speech', 'imm_symp_memory_loss', 'imm_symp_loss_of_consciousness', 'event_desc_car', 'event_desc_fall', 'event_desc_severe', 'injury_from_Accident', 'injury_from_Fall', 'injury_from_Collision', 'injury_from_Sports', 'injury_from_Assault', 'injury_from_Stroke', 'injury_from_Surgery', 'subcategory', 'severity', 'subcategory_count', 'severity_max', 'sev

In [127]:
# pofp_df.isna().sum().to_clipboard()

In [ ]:
['patient_id', 'date_of_birth', 'gender', 'patient_type', 'external_id', 'first_tbi_date', 'last_tbi_date', 
'first_tbi_from', 'first_tbi_desc', 'total_tbi', 'immediate_symptoms_resulting', 'num_head_hit_location', 
'injury_from_new', 'headhit_Not_Sure', 'headhit_Neck', 'headhit_Top_Of_Head', 'headhit_Left_Side_Of_Head',
'headhit_Front_Of_Head', 'headhit_Back_Of_Head', 'headhit_All', 'headhit_Whiplash', 'headhit_Right_Side_Of_Head',
'imm_symp_light_sensitivity', 'imm_symp_headache', 'imm_symp_dazed_or_vacant_stare', 'imm_symp_dizziness',
'imm_symp_disorientation', 'imm_symp_nausea', 'imm_symp_confusion', 'imm_symp_coma', 'imm_symp_incoherent_speech',
'imm_symp_memory_loss', 'imm_symp_loss_of_consciousness', 'event_desc_car', 'event_desc_fall',
'event_desc_severe', 'injury_from_Accident', 'injury_from_Fall', 'injury_from_Collision', 'injury_from_Sports',
'injury_from_Assault', 'injury_from_Stroke', 'injury_from_Surgery', 'subcategory', 'severity', 'subcategory_count', 
'severity_max', 'severity_min', 'severity_mean', 'severity_std', 'severity_median', 'severity_var', 'first_symptom_date', 
'last_symptom_date', 'max_severity_date', 'min_severity_date', 'day_of_max_severity', 'subcategory_count_cognitive', 
'severity_max_cognitive', 'severity_min_cognitive', 'severity_mean_cognitive', 'severity_std_cognitive', 'severity_median_cognitive', 
'severity_var_cognitive', 'first_date_cognitive', 'min_date_cognitive', 'max_date_cognitive', 'last_date_cognitive', 
'subcategory_count_emotional', 'severity_max_emotional', 'severity_min_emotional', 'severity_mean_emotional', 
'severity_std_emotional', 'severity_median_emotional', 'severity_var_emotional', 'first_date_emotional', 'min_date_emotional',
'max_date_emotional', 'last_date_emotional', 'subcategory_count_physical', 'severity_max_physical', 'severity_min_physical', 
'severity_mean_physical', 'severity_std_physical', 'severity_median_physical', 'severity_var_physical', 'first_date_physical', 
'min_date_physical', 'max_date_physical', 'last_date_physical', 'subcategory_count_sleep', 'severity_max_sleep', 'severity_min_sleep', 
'severity_mean_sleep', 'severity_std_sleep', 'severity_median_sleep', 'severity_var_sleep', 'first_date_sleep', 'min_date_sleep',
'max_date_sleep', 'last_date_sleep', 'subcategory_count_speech', 'severity_max_speech', 'severity_min_speech', 'severity_mean_speech',
'severity_std_speech', 'severity_median_speech', 'severity_var_speech', 'first_date_speech', 'min_date_speech', 
'max_date_speech', 'last_date_speech', 'subcategory_count_vision', 'severity_max_vision', 'severity_min_vision', 
'severity_mean_vision', 'severity_std_vision', 'severity_median_vision', 'severity_var_vision', 'first_date_vision',
'min_date_vision', 'max_date_vision', 'last_date_vision', 'note1', 'note1_score', 'note2', 'note2_score', 'note3', 'note3_score',
'note1_post_date', 'note2_post_date', 'note3_post_date']


In [117]:
sub_set_list = ['patient_id',
 'date_of_birth',
 'gender',
 'patient_type',
 'external_id',
 'first_tbi_date',
 'last_tbi_date',
 'first_tbi_from',
 'first_tbi_desc',
 'total_tbi',
 'headhit_Front_of_Head',
 'headhit_Back_of_Head',
 'headhit_Right_Side_of_Head',
 'headhit_Left_Side_of_Head',
 'headhit_Neck',
 'headhit_Top_of_Head',
 'headhit_No',
 'headhit_Whiplash',
 'headhit_Other',
 'imm_symp_Loss_of_Consciousness',
 'imm_symp_Memory_Loss',
 'imm_symp_Dazed_or_Vacant_Stare',
 'imm_symp_Confusion',
 'imm_symp_Incoherent_Speech',
 'imm_symp_Disorientation',
 'imm_symp_Headache', 
 'imm_symp_Coma',
 'imm_symp_Dizziness',
 'imm_symp_Nausea', 
 'imm_symp_Light_sensitivity',
 'event_desc_car',
'event_desc_fall',
'event_desc_severe',
    'injury_from_Accident',
    'injury_from_Fall',
    'injury_from_Collision',
    'injury_from_Sports',
    'injury_from_Assault',
    'injury_from_Stroke',
    'injury_from_Surgery',
#  'factor',
 'subcategory',
 'subcategory_count_emotional',
    'subcategory_count_cognitive',
    'subcategory_count_physical',
    'subcategory_count_sleep',
    'subcategory_count_speech',
    'subcategory_count_vision',
 'severity',
#  'symptom_count',
 'severity_max',
 'severity_min',
'severity_mean',
'severity_std',
'severity_median',
'severity_var',
 'first_symptom_date',
 'last_symptom_date',
 'max_severity_date',
 'min_severity_date',
 'note1',
 'note1_score',
 'note2',
 'note2_score',
 'note3',
 'note3_score',
 'note1_post_date',
 'note2_post_date',
 'note3_post_date']

In [129]:
#report = create_report(pofp_df[sub_set_list], title='EDA pofp_df')

In [130]:
def calculate_age(born, today = date.today()):
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

In [131]:
from datetime import datetime, date

def calculate_age(dob, ref=None):
    if ref is None:
        ref = date.today()
    return ref.year - dob.year - ((ref.month, ref.day) < (dob.month, dob.day))

# Clean and parse all relevant dates safely
def safe_parse_date(datestr):
    try:
        return pd.to_datetime(datestr, errors='coerce').date()
    except:
        return np.nan

# Drop rows with missing dates before parsing
pofp_df = pofp_df.dropna(subset=["date_of_birth", "first_tbi_date", "last_tbi_date"])

# Parse all dates safely
pofp_df["date_of_birth"] = pofp_df["date_of_birth"].apply(safe_parse_date)
pofp_df["first_tbi_date"] = pofp_df["first_tbi_date"].apply(safe_parse_date)
pofp_df["last_tbi_date"] = pofp_df["last_tbi_date"].apply(safe_parse_date)

# Now compute age columns safely
pofp_df["age"] = pofp_df["date_of_birth"].apply(lambda dob: calculate_age(dob) if pd.notnull(dob) else np.nan)
pofp_df["age_tbi"] = pofp_df.apply(lambda row: calculate_age(row["date_of_birth"], row["first_tbi_date"]), axis=1)
pofp_df["age_tbi_last"] = pofp_df.apply(lambda row: calculate_age(row["date_of_birth"], row["last_tbi_date"]), axis=1)


In [132]:
pofp_df.shape

(909, 135)

In [133]:
pofp_df.isna().sum()

patient_id                          0
date_of_birth                       0
gender                              0
patient_type                        2
external_id                       904
first_tbi_date                      1
last_tbi_date                       1
first_tbi_from                      0
first_tbi_desc                      0
total_tbi                           0
immediate_symptoms_resulting        0
num_head_hit_location               0
injury_from_new                     0
headhit_Not_Sure                    0
headhit_Neck                        0
headhit_Top_Of_Head                 0
headhit_Left_Side_Of_Head           0
headhit_Front_Of_Head               0
headhit_Back_Of_Head                0
headhit_All                         0
headhit_Whiplash                    0
headhit_Right_Side_Of_Head          0
imm_symp_light_sensitivity          0
imm_symp_headache                   0
imm_symp_dazed_or_vacant_stare      0
imm_symp_dizziness                  0
imm_symp_dis

In [134]:
pofp_df.to_csv("../data/merged/merged_df_latest.csv")

In [137]:
pofp_df.describe()

,total_tbi,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,headhit_Right_Side_Of_Head,imm_symp_light_sensitivity,imm_symp_headache,imm_symp_dazed_or_vacant_stare,imm_symp_dizziness,imm_symp_disorientation,imm_symp_nausea,imm_symp_confusion,imm_symp_coma,imm_symp_incoherent_speech,imm_symp_memory_loss,imm_symp_loss_of_consciousness,event_desc_car,event_desc_fall,event_desc_severe,injury_from_Accident,injury_from_Fall,injury_from_Collision,injury_from_Sports,injury_from_Assault,injury_from_Stroke,injury_from_Surgery,subcategory_count_cognitive,severity_max_cognitive,severity_min_cognitive,severity_mean_cognitive,severity_std_cognitive,severity_median_cognitive,severity_var_cognitive,first_date_cognitive,max_date_cognitive,subcategory_count_emotional,severity_max_emotional,severity_min_emotional,severity_mean_emotional,severity_std_emotional,severity_median_emotional,severity_var_emotional,first_date_emotional,max_date_emotional,subcategory_count_physical,severity_max_physical,severity_min_physical,severity_mean_physical,severity_std_physical,severity_median_physical,severity_var_physical,first_date_physical,max_date_physical,subcategory_count_sleep,severity_max_sleep,severity_min_sleep,severity_mean_sleep,severity_std_sleep,severity_median_sleep,severity_var_sleep,first_date_sleep,max_date_sleep,subcategory_count_speech,severity_max_speech,severity_min_speech,severity_mean_speech,severity_std_speech,severity_median_speech,severity_var_speech,first_date_speech,max_date_speech,subcategory_count_vision,severity_max_vision,severity_min_vision,severity_mean_vision,severity_std_vision,severity_median_vision,severity_var_vision,first_date_vision,max_date_vision,note1_score,note2_score,note3_score,age,age_tbi,age_tbi_last
count,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000,840.000000,374.000000,374.000000,374.000000,274.000000,374.000000,274.000000,0,0,729.000000,446.000000,446.000000,446.000000,305.000000,446.000000,305.000000,0,0,736.000000,305.000000,305.000000,305.000000,217.000000,305.000000,217.000000,0,0,712.000000,248.000000,248.000000,248.000000,159.000000,248.000000,159.000000,0,0,498.000000,101.000000,101.000000,101.000000,73.000000,101.000000,73.000000,0,0,607.000000,139.000000,139.000000,139.000000,106.000000,139.000000,106.000000,0,0,155.000000,92.000000,64.000000,909.000000,908.000000,908.000000
mean,3.327833,0.011001,0.163916,0.158416,0.294829,0.338834,0.368537,0.012101,0.012101,0.250825,0.007701,0.038504,0.471947,0.013201,0.657866,0.019802,0.641364,0.013201,0.337734,0.542354,0.490649,0.348735,0.235424,0.059406,0.422442,0.177118,0.115512,0.089109,0.085809,0.014301,0.007701,24.728571,62.323529,28.323529,45.130225,17.109764,45.048128,414.653436,NaT,NaT,21.991770,67.513453,30.506726,48.561910,23.670085,48.418161,779.303480,NaT,NaT,29.254076,62.665574,28.327869,44.858979,17.642432,44.227869,438.680688,NaT,NaT,12.386236,67.161290,35.500000,52.132654,17.587708,52.596774,427.299150,NaT,NaT,16.933735,61.871287,28.000000,46.498996,17.084034,47.400990,441.296927,NaT,NaT,11.504119,66.589928,29.971223,48.209150,17.262847,48.025180,427.166641,NaT,NaT,-0.181960,-0.129652,-0.099216,46.917492,37.979075,38.012115
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaT,NaT,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaT,NaT,1.000000,0.000000,0.

In [141]:
pofp_df['imm_symp_coma'].value_counts()

imm_symp_coma
0.0    897
1.0     12
Name: count, dtype: int64

In [139]:
pofp_df.shape

(909, 135)

In [140]:
pofp_df.head(2)

,patient_id,date_of_birth,gender,patient_type,external_id,first_tbi_date,last_tbi_date,first_tbi_from,first_tbi_desc,total_tbi,immediate_symptoms_resulting,num_head_hit_location,injury_from_new,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,headhit_Right_Side_Of_Head,imm_symp_light_sensitivity,imm_symp_headache,imm_symp_dazed_or_vacant_stare,imm_symp_dizziness,imm_symp_disorientation,imm_symp_nausea,imm_symp_confusion,imm_symp_coma,imm_symp_incoherent_speech,imm_symp_memory_loss,imm_symp_loss_of_consciousness,event_desc_car,event_desc_fall,event_desc_severe,injury_from_Accident,injury_from_Fall,injury_from_Collision,injury_from_Sports,injury_from_Assault,injury_from_Stroke,injury_from_Surgery,subcategory,severity,subcategory_count,severity_max,severity_min,severity_mean,severity_std,severity_median,severity_var,first_symptom_date,last_symptom_date,max_severity_date,min_severity_date,day_of_max_severity,subcategory_count_cognitive,severity_max_cognitive,severity_min_cognitive,severity_mean_cognitive,severity_std_cognitive,severity_median_cognitive,severity_var_cognitive,first_date_cognitive,min_date_cognitive,max_date_cognitive,last_date_cognitive,subcategory_count_emotional,severity_max_emotional,severity_min_emotional,severity_mean_emotional,severity_std_emotional,severity_median_emotional,severity_var_emotional,first_date_emotional,min_date_emotional,max_date_emotional,last_date_emotional,subcategory_count_physical,severity_max_physical,severity_min_physical,severity_mean_physical,severity_std_physical,severity_median_physical,severity_var_physical,first_date_physical,min_date_physical,max_date_physical,last_date_physical,subcategory_count_sleep,severity_max_sleep,severity_min_sleep,severity_mean_sleep,severity_std_sleep,severity_median_sleep,severity_var_sleep,first_date_sleep,min_date_sleep,max_date_sleep,last_date_sleep,subcategory_count_speech,severity_max_speech,severity_min_speech,severity_mean_speech,severity_std_speech,severity_median_speech,severity_var_speech,first_date_speech,min_date_speech,max_date_speech,last_date_speech,subcategory_count_vision,severity_max_vision,severity_min_vision,severity_mean_vision,severity_std_vision,severity_median_vision,severity_var_vision,first_date_vision,min_date_vision,max_date_vision,last_date_vision,note1,note1_score,note2,note2_score,note3,note3_score,note1_post_date,note2_post_date,note3_post_date,age,age_tbi,age_tbi_last
0,5c96ba1a-8b2d-49bc-8e8e-b07761948286,1973-05-23,female,caregiver,NaN,2019-11-13,2019-11-13,Collision,My husband was stopped at a stoplight and was ...,1.0,"loss of consciousness,disorientation,incoheren...",6,Assault,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,"(cognitive, emotional, physical, sleep, speech...","(nan, nan, nan, nan, nan, nan)","(6, 9, 7, 6, 5, 2)","(nan, nan, nan, nan, nan, nan)","(nan, nan, nan, nan, nan, nan)","(nan, nan, nan, nan, nan, nan)","(nan, nan, nan, nan, nan, nan)","(nan, nan, nan, nan, nan, nan)","(nan, nan, nan, nan, nan, nan)","(NaT, NaT, NaT, NaT, NaT, NaT)","(11/12/2019 0:00, 11/12/2019 0:00, 11/12/2019 ...","(NaT, NaT, NaT, NaT, NaT, NaT)","(nan, nan, nan, nan, nan, nan)","(nan, nan, nan, nan, nan, nan)",6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,11/12/2019 0:00,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,11/12/2019 0:00,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,11/12/2019 0:00,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,11/12/2019 0:00,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,11/12/2019 0:00,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,11/12/2019 0:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52,46.0,46.0
1,eda39327-b38f-41de-a46a-8782787369b7,1991-07-10,male,caregiver,NaN,2020-05-05,2020-05-05,Accident,My 2 yr old son drowned,1.0,"disorientation,incoherent speech,confusion,mem...",1,Accident,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,